# How to Save Bokeh Plots

This guide shows how to save hvPlot plots created with the Bokeh backend (the default) as interactive HTML files or as static PNG images.

:::{tip}
Not sure whether to save with Bokeh or Matplotlib? See [How to Choose a Backend for Saving Plots](choose_backend_for_saving_plots.md).
:::

## Save as an interactive HTML file

HTML is the format of choice for Bokeh plots, as it keeps all the interactive features like zooming, panning and hover tooltips:

In [ ]:
import hvplot.pandas  # noqa

penguins = hvplot.sampledata.penguins("pandas")

plot = penguins.hvplot.scatter(
    x='bill_length_mm', y='bill_depth_mm',
    color='species', title="Penguin Bill Dimensions"
)
plot

In [ ]:
hvplot.save(plot, 'penguins.html')

## Make the HTML file work offline

By default, the HTML file loads the BokehJS JavaScript library from a CDN, so it needs an internet connection to be displayed. The `resources` argument controls this:

- `'cdn'` (default): load the JavaScript from the CDN, small file but requires internet access.
- `'inline'`: embed the JavaScript in the file, so it works offline and in airgapped environments.
- A [`bokeh.resources.Resources`](https://docs.bokeh.org/en/latest/docs/reference/resources.html) object, such as `bokeh.resources.CDN` or `bokeh.resources.INLINE`.

In [ ]:
hvplot.save(plot, 'penguins_inline.html', resources='inline')

Embedding the JavaScript makes the file much larger, around 3 MB instead of a few tens of kB:

In [ ]:
from pathlib import Path

for filename in ['penguins.html', 'penguins_inline.html']:
    print(f"{filename}: {Path(filename).stat().st_size / 1e6:.2f} MB")

## Save as a PNG image

Bokeh plots are rendered by the browser, so exporting them to PNG requires a headless browser. Since Bokeh 3.10, the simplest option is [Playwright](https://playwright.dev/python/), which downloads and manages the browser for you. Install Playwright in your virtual environment (with pip or uv or conda or pixi ...). Then run this command to install the Chromium browser that Playwright uses to generate static plots:

```bash
playwright install --only-shell chromium
```

Once set up, `hvplot.save` uses it automatically and removes the toolbar from the image:

In [ ]:
hvplot.save(plot, 'penguins.png')

:::{note}
With Bokeh older than 3.10, PNG export requires Selenium and a web driver (geckodriver or chromedriver), see the [Bokeh export documentation](https://docs.bokeh.org/en/latest/docs/user_guide/output/export.html).
:::

## Increase the PNG resolution

`hvplot.save` produces one image pixel per screen pixel, so the plot above is saved as a 700x300 image. Increasing `width` and `height` gives a larger image, but the text and markers keep the same size and look smaller. To get a sharper image with the same layout, render the plot to a Bokeh object with `hv.render` and export it with Bokeh's `export_png` and its `scale_factor` argument:

In [ ]:
import holoviews as hv
from bokeh.io import export_png

export_png(hv.render(plot.opts(toolbar=None)), filename='penguins_2x.png', scale_factor=2)

This saves a 1400x600 image of the same plot.

:::{seealso}
- [How to Save Matplotlib Plots](save_matplotlib_plots.ipynb) to create static images without a browser.
- [How to Choose a Backend for Saving Plots](choose_backend_for_saving_plots.md).
- [How to Customize the Underlying Figure](customize_underlying_figure.ipynb), to edit the Bokeh figure returned by `hv.render`.
:::

:::{seealso}
- [Display plots](display_plots.ipynb) in notebooks, scripts and apps
- [Load and switch plotting extensions](load_switch_plotting_extensions.ipynb)
:::

In [ ]:
from pathlib import Path

for filename in ['penguins.html', 'penguins_inline.html', 'penguins.png', 'penguins_2x.png']:
    Path(filename).unlink(missing_ok=True)